In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================
# 여기서부터는 이제 wl_diff 변수가 표준임

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_train_fe_wl.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe_wl.parquet')
from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [4]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]

print(len(wl_cols), len(wl_diff_cols), len(rain_cols), len(env_cols), len(num_cols))

6 6 9 7 27


In [5]:
# ============================================================
# 3. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [8]:
# ============================================================
# 4. station_id 원핫인코딩
# ============================================================
# OHE 포함한게 조금이나마 나은 결과, 포함해서 모델링 시행

from sklearn.preprocessing import OneHotEncoder

# 원핫인코더 생성
ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])
# 여기까지는 일반적은 인코딩, 독립변수에 붙이는 작업은 모델링에서 한번에.

In [ ]:
# ============================================================
# 5. 모델3: 수위, 강우, 유역환경 변수군 ablation 모델 만들기
# ============================================================

# 5-1. 수위, 강우, 유역환경 변수군 ablation 데이터셋 컬럼 정의

sets_ablation_cols = {
    'no_wl_cols' : [col for col in num_cols if col not in wl_cols],
    'no_wl_diff_cols' : [
        col for col in num_cols if col not in [
            col for col in wl_diff_cols if col != 'wl_t'
        ]   
    ],
    'no_rain_cols' : [col for col in num_cols if col not in rain_cols],
    'no_env_cols' : [col for col in num_cols if col not in env_cols]
}

for name, cols in sets_ablation_cols.items():  # datasets의 이름과 내용물이 같이 나옴(반환값 두개)
    print(name, len(cols), cols)  # len()으로 각각 21, 22, 18, 20가 나와야 함
    print()

no_wl_cols 21 ['imerg_rain_1h', 'imerg_rain_6h', 'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h', 'radar_reflectivity_mean', 'radar_reflectivity_p95', 'radar_reflectivity_ge20_fraction', 'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio', 'stream_order', 'wl_diff_1h', 'wl_diff_3h', 'wl_diff_6h', 'wl_diff_12h', 'wl_diff_24h']

no_wl_diff_cols 22 ['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h', 'wl_t_minus_12h', 'wl_t_minus_24h', 'imerg_rain_1h', 'imerg_rain_6h', 'imerg_rain_24h', 'sfc_rain_1h', 'sfc_rain_6h', 'sfc_rain_24h', 'radar_reflectivity_mean', 'radar_reflectivity_p95', 'radar_reflectivity_ge20_fraction', 'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio', 'stream_order']

no_rain_cols 18 ['wl_t', 'wl_t_minus_1h', 'wl_t_minus_3h', 'wl_t_minus_6h', 'wl_t_minus_12h', 'wl_t_minus_24h', 'cat_area', 'wsarea', 'mean_elevation', 'mean_slope', 'urban_ratio', 'forest_ratio', 'stream_order', 'wl_dif

In [13]:
# 5-2 모델링: 변수군 3종 ablation
# ohe와 합치기~모델링~평가 및 검증까지 for 문안에 한번에 만들기

# 3-1. 변수군별 제외 모델링

result = {}  # 조건별 rmse 저장용 딕셔너리

# 해당 데이터셋과 ohe 합치기
from scipy.sparse import hstack

for name, cols in sets_ablation_cols.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[cols],
        train_st_id_ohe
    ]).tocsr()
    
    X_val = hstack([
        val[cols],
        val_st_id_ohe
    ]).tocsr()


    # 모델링
    from lightgbm import LGBMRegressor

    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)  # X_train은 for문에서 y_train은 상단 셀에서 가져옴

    pred = model.predict(X_val)  # X_val은 for문에서 가져옴


    # 성능평가

    from sklearn.metrics import root_mean_squared_error
    rmse = root_mean_squared_error(y_val, pred)  # y_val은 상단 셀에서 가져옴

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨
    print()

print(result)

no_wl_cols
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005347 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6205
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 486
[LightGBM] [Info] Start training from score 0.111701

no_wl_diff_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026547 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6460
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 487
[LightGBM] [Info] Start training from score 0.111701

no_rain_cols
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019068 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5477
[LightGBM]

In [ ]:
# ============================================================
# 6. 모델3: 수위 개별변수별 ablation 모델링
# ============================================================

wl_ablation_col = {
    'no_wl_t' : 'wl_t',
    'no_wl_t_minus_1h' : 'wl_t_minus_1h',
    'no_wl_t_minus_3h' : 'wl_t_minus_3h',
    'no_wl_t_minus_6h' : 'wl_t_minus_6h',
    'no_wl_t_minus_12h' : 'wl_t_minus_12h',
    'no_wl_t_minus_24h' : 'wl_t_minus_24h',
}


for name, col in wl_ablation_col.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[[c for c in num_cols if c != col]],
        train_st_id_ohe
    ]).tocsr()
    print(f"train에 들어갈 컬럼 개수는? {len([c for c in num_cols if c != col])}개")
    
    X_val = hstack([
        val[[c for c in num_cols if c != col]],
        val_st_id_ohe
    ]).tocsr()
    print(f"train에 들어갈 컬럼 개수는? {len([c for c in num_cols if c != col])}개")

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

print(result)

no_wl_t
train에 들어갈 컬럼 개수는? 26개
train에 들어갈 컬럼 개수는? 26개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.058411 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7480
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 491
[LightGBM] [Info] Start training from score 0.111701
no_wl_t_minus_1h
train에 들어갈 컬럼 개수는? 26개
train에 들어갈 컬럼 개수는? 26개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.108343 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7480
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 491
[LightGBM] [Info] Start training from score 0.111701
no_wl_t_minus_3h
train에 들어갈 컬럼 개수는? 26개
train에 들어갈 컬럼 개수는? 26개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.083590 seconds.
You can set `force_col_wise=

In [15]:
# ============================================================
# 7. 모델3: wl_differ(수위차분) 개별변수별 ablation 모델링
# ============================================================

wl_diff_ablation_col = {
    'no_wl_diff_1h' : 'wl_diff_1h',
    'no_wl_diff_3h' : 'wl_diff_3h',
    'no_wl_diff_6h' : 'wl_diff_6h',
    'no_wl_diff_12h' : 'wl_diff_12h',
    'no_wl_diff_24h' : 'wl_diff_24h',
}

for name, col in wl_diff_ablation_col.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[[c for c in num_cols if c != col]],
        train_st_id_ohe
    ]).tocsr()
    print(f"train에 들어갈 컬럼 개수는? {len([c for c in num_cols if c != col])}개")
    
    X_val = hstack([
        val[[c for c in num_cols if c != col]],
        val_st_id_ohe
    ]).tocsr()
    print(f"train에 들어갈 컬럼 개수는? {len([c for c in num_cols if c != col])}개")

    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

print(result)

no_wl_diff_1h
train에 들어갈 컬럼 개수는? 26개
train에 들어갈 컬럼 개수는? 26개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026979 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7480
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 491
[LightGBM] [Info] Start training from score 0.111701
no_wl_diff_3h
train에 들어갈 컬럼 개수는? 26개
train에 들어갈 컬럼 개수는? 26개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029310 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7480
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 491
[LightGBM] [Info] Start training from score 0.111701
no_wl_diff_6h
train에 들어갈 컬럼 개수는? 26개
train에 들어갈 컬럼 개수는? 26개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033279 seconds.
You can set `force_col_wise=

In [16]:
# ============================================================
# 7. 모델3: 강우 세부변수군(imerg, sfc, radar) ablation 모델링
# ============================================================
# 3-3. 강우변수 세분화해서 영향력 확인
rain_ablation_cols = {
    'no_imerg' : [
        'imerg_rain_1h',
        'imerg_rain_6h',
        'imerg_rain_24h'
    ],
    'no_sfc' : [
        'sfc_rain_1h',
        'sfc_rain_6h',
        'sfc_rain_24h'
    ],
    'no_radar_reflect' : [
        'radar_reflectivity_mean',
        'radar_reflectivity_p95',
        'radar_reflectivity_ge20_fraction'
    ]
}

for name, cols in rain_ablation_cols.items(): # name: 변수군의 이름, cols: 컬럼군
    print(name)
    X_train = hstack([
        train[[c for c in num_cols if c not in cols]],
        train_st_id_ohe
    ]).tocsr()
    print(f"train에 들어갈 컬럼 개수는? {len([c for c in num_cols if c not in cols])}개")
    
    X_val = hstack([
        val[[c for c in num_cols if c not in cols]],
        val_st_id_ohe
    ]).tocsr()
    print(f"val에 들어갈 컬럼 개수는? {len([c for c in num_cols if c not in cols])}개")
    
    # 모델링
    model = LGBMRegressor(
        random_state = 42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    # 성능평가
    rmse = root_mean_squared_error(y_val, pred)

    # 결과에 더하기
    result[name] = round(rmse, 4)  # result는 딕셔너리, []는 key가 됨

no_imerg
train에 들어갈 컬럼 개수는? 24개
val에 들어갈 컬럼 개수는? 24개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024741 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6970
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 489
[LightGBM] [Info] Start training from score 0.111701
no_sfc
train에 들어갈 컬럼 개수는? 24개
val에 들어갈 컬럼 개수는? 24개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.066644 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7002
[LightGBM] [Info] Number of data points in the train set: 616331, number of used features: 489
[LightGBM] [Info] Start training from score 0.111701
no_radar_reflect
train에 들어갈 컬럼 개수는? 24개
val에 들어갈 컬럼 개수는? 24개
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.063952 seconds.
You can set `force_col_wise=true` to remove

In [18]:
# rmse 결과 df로 보기

result_df = pd.DataFrame(
result.items(),
columns=['model', 'rmse']
)

result_df['from_base_model'] = round((result_df['rmse'] - 0.2019), 4)

result_df

,model,rmse,from_base_model
0,no_wl_cols,0.2046,0.0027
1,no_wl_diff_cols,0.2133,0.0114
2,no_rain_cols,0.2120,0.0101
3,no_env_cols,0.2070,0.0051
4,no_wl_t,0.2023,0.0004
5,no_wl_t_minus_1h,0.2020,0.0001
6,no_wl_t_minus_3h,0.2023,0.0004
7,no_wl_t_minus_6h,0.2024,0.0005
8,no_wl_t_minus_12h,0.2018,-0.0001
9,no_wl_t_minus_24h,0.2016,-0.0003
